# 01 — Data profiling

## Pertanyaan

Seberapa lengkap, konsisten, dan siap dianalisis data World Bank, Bank Indonesia, dan BPS yang saat ini tersedia di analytical marts?

## Metode

Notebook membaca empat mart langsung dari MariaDB, menghitung jumlah baris, rentang periode, tipe data, nilai unik, missing value, serta statistik deskriptif. Tidak ada nilai observasi yang diimputasi.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from analytics.descriptive.data_access import DATASET_SOURCES, dataset_inventory
from analytics.descriptive.notebook_support import insight, prepare_notebook, save_figure
from analytics.descriptive.statistics import descriptive_statistics, profile_frame

data = prepare_notebook()
inventory = dataset_inventory(data)
display(inventory)

## Hasil

In [ ]:
profiles = {}
for dataset_name in ("national", "asean", "monetary", "regional"):
    profiles[dataset_name] = profile_frame(getattr(data, dataset_name))
    print(f"Profil: {dataset_name}")
    display(profiles[dataset_name])

national_columns = [
    "gdp_growth_percent",
    "inflation_percent",
    "unemployment_percent",
    "population",
    "gdp_per_capita_current_usd",
]
national_statistics = descriptive_statistics(data.national, national_columns)
display(national_statistics)

In [ ]:
missing_summary = pd.concat(
    [profile.assign(dataset=name) for name, profile in profiles.items()],
    ignore_index=True,
)
missing_summary["label"] = missing_summary["dataset"] + "." + missing_summary["column"]
plot_data = missing_summary.sort_values("missing_percent", ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_data["label"], plot_data["missing_percent"])
ax.invert_yaxis()
ax.set(title="Dua puluh kolom dengan missing value tertinggi", xlabel="Missing (%)")
save_figure(fig, "01_missing_profile.png")
display(fig)
plt.close(fig)

In [ ]:
complete_years = int(data.national[national_columns].notna().all(axis=1).sum())
display(insight(
    f"Mart nasional memiliki {len(data.national)} tahun observasi; {complete_years} tahun lengkap untuk lima KPI utama.",
    frame=data.national,
    source=DATASET_SOURCES["national"],
    limitation="Kelengkapan kolom menunjukkan kesiapan analisis, tetapi tidak menjamin bahwa angka sumber bebas dari revisi.",
))
regional_finding = (
    f"Mart regional berisi {len(data.regional)} observasi resmi."
    if not data.regional.empty
    else "Mart regional belum berisi observasi karena pipeline produksi BPS belum dijalankan dengan API key."
)
display(insight(
    regional_finding,
    frame=data.regional,
    source=DATASET_SOURCES["regional"],
    limitation="Ketiadaan data dilaporkan apa adanya; notebook tidak membuat fallback sintetis.",
))

## Interpretasi

Profil ini menentukan analisis mana yang sah dilakukan sekarang. Dataset kosong tetap ditampilkan sebagai status ketersediaan, bukan dianggap bernilai nol.

## Keterbatasan

Profil hanya memeriksa struktur, missing value, dan distribusi dasar. Validasi definisi indikator, revisi historis, serta keterbandingan lintas sumber tetap mengikuti metadata dan dokumentasi masing-masing sumber.